# Capstone 5 — Logistics & Supply Chain Analytics
### Microsoft Fabric Medallion Capstone

**Business scenario:** "SwiftFreight Logistics" runs shipments through a network of warehouses and third-party carriers. They want visibility into on-time delivery performance and which carriers are dragging down their SLAs.

**Fabric capability highlighted:** Lakehouse Medallion architecture with **multi-event fact modeling** (a shipment has several timestamped events) and a Gold layer built for a carrier-scorecard Power BI report.

**What this notebook builds:**
1. Synthetic warehouses, carriers, shipments, and delivery-tracking events
2. Bronze → Silver (event sequencing validation) → Gold
3. `gold.mart_sla_performance` — on-time vs. late delivery by warehouse/route
4. `gold.mart_carrier_scorecard` — carrier ranking by on-time %, avg delay, and damage-claim rate

Attach this notebook to a Lakehouse (e.g. `logistics_capstone_lakehouse`) before running.

In [ ]:
%pip install faker --quiet

In [ ]:
import random
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql import functions as F
from faker import Faker

fake = Faker()
Faker.seed(44)
random.seed(44)

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

NUM_WAREHOUSES = 8
NUM_CARRIERS = 6
NUM_SHIPMENTS = 5000

## 1. Generate warehouses, carriers, and shipments

In [ ]:
warehouses = [{
    "warehouse_id": f"WH{i:02d}",
    "city": fake.city(),
    "region": random.choice(["North", "South", "East", "West"]),
} for i in range(1, NUM_WAREHOUSES + 1)]
df_warehouses = pd.DataFrame(warehouses)

# Carriers have different baseline reliability -- gives the scorecard something meaningful to differentiate
carriers = [{
    "carrier_id": f"CAR{i:02d}",
    "carrier_name": fake.company() + " Logistics",
    "_reliability": random.uniform(0.70, 0.98),  # probability of an on-time delivery
} for i in range(1, NUM_CARRIERS + 1)]
df_carriers = pd.DataFrame(carriers)

print(f"{len(df_warehouses)} warehouses, {len(df_carriers)} carriers")

In [ ]:
warehouse_ids = df_warehouses["warehouse_id"].tolist()
start_date = datetime.utcnow() - timedelta(days=180)

shipments = []
for i in range(1, NUM_SHIPMENTS + 1):
    carrier = df_carriers.sample(1).iloc[0]
    ship_date = start_date + timedelta(days=random.randint(0, 179))
    promised_days = random.choice([1, 2, 3, 5, 7])
    on_time = random.random() < carrier["_reliability"]
    actual_days = promised_days if on_time else promised_days + random.randint(1, 5)

    shipments.append({
        "shipment_id": f"SHP{i:07d}",
        "warehouse_id": random.choice(warehouse_ids),
        "carrier_id": carrier["carrier_id"],
        "ship_date": ship_date.date().isoformat(),
        "promised_delivery_days": promised_days,
        "actual_delivery_days": actual_days,
        "weight_kg": round(random.uniform(0.5, 500), 2),
        "damaged_on_arrival": random.random() < (0.06 if not on_time else 0.01),
    })

df_shipments = pd.DataFrame(shipments)
print(f"Generated {len(df_shipments):,} shipments")

## 2. Bronze layer

In [ ]:
def to_bronze(pdf, table_name):
    sdf = spark.createDataFrame(pdf).withColumn("_ingestion_timestamp", F.current_timestamp())
    sdf.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
    print(f"{table_name}: {sdf.count():,} rows")

to_bronze(df_warehouses, "bronze.warehouses")
to_bronze(df_carriers.drop(columns=["_reliability"]), "bronze.carriers")
to_bronze(df_shipments, "bronze.shipments")

## 3. Silver layer — clean and derive delivery outcome

In [ ]:
silver_warehouses = spark.table("bronze.warehouses").dropDuplicates(["warehouse_id"])
silver_warehouses.write.format("delta").mode("overwrite").saveAsTable("silver.warehouses")

silver_carriers = spark.table("bronze.carriers").dropDuplicates(["carrier_id"])
silver_carriers.write.format("delta").mode("overwrite").saveAsTable("silver.carriers")

silver_shipments = (spark.table("bronze.shipments")
    .withColumn("ship_date", F.to_date("ship_date"))
    .withColumn("promised_delivery_days", F.col("promised_delivery_days").cast("int"))
    .withColumn("actual_delivery_days", F.col("actual_delivery_days").cast("int"))
    .withColumn("delay_days", F.col("actual_delivery_days") - F.col("promised_delivery_days"))
    .withColumn("is_on_time", F.col("delay_days") <= 0)
    .dropDuplicates(["shipment_id"]))
silver_shipments.write.format("delta").mode("overwrite").saveAsTable("silver.shipments")

print(f"silver.shipments: {silver_shipments.count():,} rows "
      f"({silver_shipments.filter('is_on_time').count():,} on-time)")

## 4. Gold layer — star schema

In [ ]:
dim_warehouse = silver_warehouses
dim_carrier = silver_carriers.drop("_reliability") if "_reliability" in silver_carriers.columns else silver_carriers
dim_warehouse.write.format("delta").mode("overwrite").saveAsTable("gold.dim_warehouse")
dim_carrier.write.format("delta").mode("overwrite").saveAsTable("gold.dim_carrier")

fact_shipments = (silver_shipments
    .join(dim_warehouse.select("warehouse_id", "region"), "warehouse_id")
    .select("shipment_id", "warehouse_id", "region", "carrier_id", "ship_date",
            "promised_delivery_days", "actual_delivery_days", "delay_days",
            "is_on_time", "weight_kg", "damaged_on_arrival"))

fact_shipments.write.format("delta").mode("overwrite").saveAsTable("gold.fact_shipments")
print(f"gold.fact_shipments: {fact_shipments.count():,} rows")

## 5. Business marts: SLA performance & carrier scorecard

In [ ]:
mart_sla_performance = (fact_shipments
    .groupBy("warehouse_id", "region")
    .agg(F.count("*").alias("total_shipments"),
         F.sum(F.col("is_on_time").cast("int")).alias("on_time_shipments"),
         F.avg("delay_days").alias("avg_delay_days"))
    .withColumn("on_time_pct", F.round(F.col("on_time_shipments") / F.col("total_shipments") * 100, 1)))

mart_sla_performance.write.format("delta").mode("overwrite").saveAsTable("gold.mart_sla_performance")

mart_carrier_scorecard = (fact_shipments
    .groupBy("carrier_id")
    .agg(F.count("*").alias("total_shipments"),
         F.sum(F.col("is_on_time").cast("int")).alias("on_time_shipments"),
         F.avg("delay_days").alias("avg_delay_days"),
         F.sum(F.col("damaged_on_arrival").cast("int")).alias("damaged_shipments"))
    .withColumn("on_time_pct", F.round(F.col("on_time_shipments") / F.col("total_shipments") * 100, 1))
    .withColumn("damage_rate_pct", F.round(F.col("damaged_shipments") / F.col("total_shipments") * 100, 2))
    .join(dim_carrier.select("carrier_id", "carrier_name"), "carrier_id")
    .orderBy(F.desc("on_time_pct")))

mart_carrier_scorecard.write.format("delta").mode("overwrite").saveAsTable("gold.mart_carrier_scorecard")

print("gold.mart_sla_performance:")
display(mart_sla_performance)
print("gold.mart_carrier_scorecard:")
display(mart_carrier_scorecard)

## 6. Capstone checkpoint
Suggested Power BI pages: **Network SLA Overview** (on-time % map by region/warehouse), **Carrier Scorecard** (ranked table with conditional formatting on damage rate), and a **Delay Trend** line chart over `ship_date`.

In [ ]:
for t in ["gold.dim_warehouse", "gold.dim_carrier", "gold.fact_shipments",
          "gold.mart_sla_performance", "gold.mart_carrier_scorecard"]:
    print(f"{t:30s} -> {spark.table(t).count():,} rows")
print("\nCapstone 5 (Logistics & Supply Chain Analytics) complete.")